# Session 2 – K-Means Clustering from Scratch

## 1. Food Delivery Locations – Plot, Pick Centroids & Assign Manually
Delivery points: `[2,3], [5,8], [1,2], [6,9], [7,7]`

Two initial centroids are picked **randomly from the data points**. In this run we use:
**C1 = [1, 2]** and **C2 = [7, 7]**.

In [ ]:
import math
import matplotlib.pyplot as plt

points = [[2, 3], [5, 8], [1, 2], [6, 9], [7, 7]]

# Randomly picked initial centroids (chosen from the data points)
# e.g. random.sample(points, 2) -> here fixed for a reproducible manual walkthrough
centroids = [[1, 2], [7, 7]]

# --- Plot the points and the initial centroids ---
xs, ys = zip(*points)
plt.figure(figsize=(6, 5))
plt.scatter(xs, ys, s=100, c="steelblue", label="Delivery locations")
for p in points:
    plt.annotate(f"({p[0]},{p[1]})", (p[0] + 0.15, p[1] + 0.15))
cx, cy = zip(*centroids)
plt.scatter(cx, cy, s=300, c="red", marker="X", label="Initial centroids")
plt.title("Food Delivery Locations")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.show()

### Manual Euclidean distance calculation
$d(p, c) = \sqrt{(x_p - x_c)^2 + (y_p - y_c)^2}$

| Point | Distance to C1 [1,2] | Distance to C2 [7,7] | Assigned Cluster |
|---|---|---|---|
| [2,3] | √(1² + 1²) = √2 = **1.414** | √(5² + 4²) = √41 = 6.403 | **C1** |
| [5,8] | √(4² + 6²) = √52 = 7.211 | √(2² + 1²) = √5 = **2.236** | **C2** |
| [1,2] | √(0² + 0²) = **0.000** | √(6² + 5²) = √61 = 7.810 | **C1** |
| [6,9] | √(5² + 7²) = √74 = 8.602 | √(1² + 2²) = √5 = **2.236** | **C2** |
| [7,7] | √(6² + 5²) = √61 = 7.810 | √(0² + 0²) = **0.000** | **C2** |

**Result:** Cluster 1 = {[2,3], [1,2]} (south-west area), Cluster 2 = {[5,8], [6,9], [7,7]} (north-east area).

In [ ]:
# Verify the manual calculation with code
for p in points:
    d1 = math.dist(p, centroids[0])
    d2 = math.dist(p, centroids[1])
    cluster = "C1" if d1 < d2 else "C2"
    print(f"Point {p}: d(C1)={d1:.3f}, d(C2)={d2:.3f} -> {cluster}")

# Plot the assignment
colors = ["tab:green" if math.dist(p, centroids[0]) < math.dist(p, centroids[1]) else "tab:orange" for p in points]
plt.figure(figsize=(6, 5))
plt.scatter(xs, ys, s=100, c=colors)
plt.scatter(cx, cy, s=300, c="red", marker="X")
plt.title("Points assigned to nearest centroid (green=C1, orange=C2)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

## 2. `assign_clusters(points, centroids)` – Euclidean Distance
Returns a list where the i-th value is the index of the closest centroid for the i-th point.

In [ ]:
def euclidean_distance(p, q):
    return math.sqrt(sum((a - b) ** 2 for a, b in zip(p, q)))


def assign_clusters(points, centroids):
    """Assign each point to the nearest centroid using Euclidean distance."""
    assignments = []
    for p in points:
        distances = [euclidean_distance(p, c) for c in centroids]
        assignments.append(distances.index(min(distances)))
    return assignments


print(assign_clusters(points, centroids))   # expected: [0, 1, 0, 1, 1]

## 3. Support for Manhattan Distance
Manhattan distance: $d(p, c) = |x_p - x_c| + |y_p - y_c|$ — the distance a delivery rider travels along a city grid.
The user chooses the metric via the `metric` parameter (`'euclidean'` or `'manhattan'`).

In [ ]:
def manhattan_distance(p, q):
    return sum(abs(a - b) for a, b in zip(p, q))


def assign_clusters(points, centroids, metric="euclidean"):
    """Assign each point to the nearest centroid.

    metric: 'euclidean' (straight-line) or 'manhattan' (city-block).
    """
    metric = metric.lower()
    if metric == "euclidean":
        dist_fn = euclidean_distance
    elif metric == "manhattan":
        dist_fn = manhattan_distance
    else:
        raise ValueError("metric must be 'euclidean' or 'manhattan'")

    assignments = []
    for p in points:
        distances = [dist_fn(p, c) for c in centroids]
        assignments.append(distances.index(min(distances)))
    return assignments


print("Euclidean :", assign_clusters(points, centroids, metric="euclidean"))
print("Manhattan :", assign_clusters(points, centroids, metric="manhattan"))

# Example where the two metrics disagree
test_points = [[0, 0]]
test_centroids = [[3, 3], [5, 0]]
print("\nPoint [0,0] with centroids [3,3] and [5,0]:")
print("  Euclidean distances:", [round(euclidean_distance(test_points[0], c), 3) for c in test_centroids],
      "->", assign_clusters(test_points, test_centroids, "euclidean"))
print("  Manhattan distances:", [manhattan_distance(test_points[0], c) for c in test_centroids],
      "->", assign_clusters(test_points, test_centroids, "manhattan"))

## 4. `update_centroids(points, assignments, k)`
The new centroid of each cluster = mean of all points assigned to it.
If a cluster ends up empty, we return `None` for it (it can be re-initialised later).

In [ ]:
def update_centroids(points, assignments, k):
    """Return the new centroid (mean of assigned points) for each of the k clusters."""
    dims = len(points[0])
    sums = [[0.0] * dims for _ in range(k)]
    counts = [0] * k

    for p, cluster in zip(points, assignments):
        counts[cluster] += 1
        for d in range(dims):
            sums[cluster][d] += p[d]

    new_centroids = []
    for i in range(k):
        if counts[i] == 0:
            new_centroids.append(None)          # empty cluster
        else:
            new_centroids.append([round(s / counts[i], 3) for s in sums[i]])
    return new_centroids


# Test 1: delivery points from Q1
assignments = assign_clusters(points, centroids)
print("Assignments  :", assignments)
print("New centroids:", update_centroids(points, assignments, k=2))
# Cluster 0 -> mean of [2,3],[1,2]        = [1.5, 2.5]
# Cluster 1 -> mean of [5,8],[6,9],[7,7]  = [6.0, 8.0]

In [ ]:
# Test 2: a bigger sample dataset - run full K-Means iterations using our own functions
sample = [[1, 1], [1.5, 2], [3, 4], [5, 7], [3.5, 5], [4.5, 5], [3.5, 4.5], [9, 9], [8.5, 9.5], [9.5, 8]]
k = 3
current = [sample[0], sample[3], sample[7]]   # initial centroids

for it in range(1, 11):
    labels = assign_clusters(sample, current)
    new = update_centroids(sample, labels, k)
    print(f"Iteration {it}: labels={labels}, centroids={new}")
    if new == current:
        print("Converged!")
        break
    current = new

# Visualise final clusters
sx, sy = zip(*sample)
plt.figure(figsize=(6, 5))
plt.scatter(sx, sy, c=labels, cmap="viridis", s=100)
fx, fy = zip(*current)
plt.scatter(fx, fy, c="red", marker="X", s=300, label="Final centroids")
plt.title("K-Means from scratch (k=3)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

## 5. Elbow Method – Zomato Restaurants (Rating vs Average Price for Two)
We create a dataset of 60 restaurants with two features:
- **rating** (1–5 stars)
- **price_for_two** (₹)

We run K-Means for k = 1 … 6 and plot the **Within-Cluster Sum of Squares (WCSS / inertia)**.
The "elbow" is the k after which adding more clusters gives only a small drop in WCSS.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)

# Three natural restaurant types
budget  = np.column_stack([rng.normal(3.4, 0.25, 20), rng.normal(300, 60, 20)])    # street food / QSR
casual  = np.column_stack([rng.normal(4.0, 0.20, 20), rng.normal(900, 120, 20)])   # casual dining
premium = np.column_stack([rng.normal(4.6, 0.15, 20), rng.normal(2500, 300, 20)])  # fine dining

data = np.vstack([budget, casual, premium])
data[:, 0] = np.clip(data[:, 0], 1, 5).round(1)
data[:, 1] = data[:, 1].round(-1)

df = pd.DataFrame(data, columns=["rating", "price_for_two"])
print(df.describe().round(2))
df.head()

In [ ]:
# Scale features (price is in hundreds/thousands, rating is 1-5)
X = StandardScaler().fit_transform(df[["rating", "price_for_two"]])

wcss = []
k_values = range(1, 7)
for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    wcss.append(km.inertia_)
    print(f"k={k}: WCSS = {km.inertia_:.2f}")

# Find the elbow: first k after which adding one more cluster
# reduces WCSS by less than 5% of the total (k=1) WCSS
improvements = -np.diff(wcss) / wcss[0]
for i, imp in enumerate(improvements):
    print(f"  k={i+1} -> k={i+2}: WCSS reduced by {imp*100:.1f}% of total")
elbow_k = int(np.argmax(improvements < 0.05)) + 1
print(f"\nElbow point detected at k = {elbow_k}")

plt.figure(figsize=(7, 5))
plt.plot(list(k_values), wcss, "o-", color="royalblue", linewidth=2, markersize=8)
plt.scatter(elbow_k, wcss[elbow_k - 1], s=300, facecolors="none", edgecolors="red", linewidths=2)
plt.annotate(f"Elbow (k={elbow_k})", (elbow_k, wcss[elbow_k - 1]),
             xytext=(elbow_k + 0.5, wcss[elbow_k - 1] + max(wcss) * 0.15),
             arrowprops=dict(arrowstyle="->", color="red"), color="red", fontsize=12)
plt.title("Elbow Method - Zomato Restaurants")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Within-Cluster Sum of Squares (WCSS)")
plt.xticks(list(k_values))
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

In [ ]:
# Visualise the clusters using the optimal k
km = KMeans(n_clusters=elbow_k, random_state=42, n_init=10)
df["cluster"] = km.fit_predict(X)

plt.figure(figsize=(7, 5))
scatter = plt.scatter(df["price_for_two"], df["rating"], c=df["cluster"], cmap="viridis", s=70)
plt.title(f"Zomato Restaurants clustered with k={elbow_k}")
plt.xlabel("Average price for two (Rs.)")
plt.ylabel("Rating")
plt.legend(*scatter.legend_elements(), title="Cluster")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

print(df.groupby("cluster")[["rating", "price_for_two"]].mean().round(2))

### Interpretation
- WCSS drops sharply from **k=1 → k=2 → k=3**, then flattens for k=4, 5, 6.
- The bend ("elbow") is at **k = 3**, so 3 clusters is the optimal choice.
- The 3 clusters correspond to meaningful restaurant segments:
  1. **Budget / Quick-service** – low price (~₹300), average rating (~3.4)
  2. **Casual dining** – mid price (~₹900), good rating (~4.0)
  3. **Fine dining** – high price (~₹2,500), excellent rating (~4.6)
- Adding more clusters beyond 3 only splits these natural groups further without much gain.